# Expert Reviews data cleaning and database preparation

This is my working Jupyter notebook for cleaning the Expert Reviews data and preparing the `CRITICS` and `CRITIC_REVIEWS` tables from my ERD.

I keep the code fairly simple because I am still learning Python. I use comments in the code to explain what I am doing, and I keep the outputs from my checks so I can see the results I actually got while cleaning the data.

The separate explanatory notebook gives the longer explanation of the overall process. This notebook is the practical record of the Python work and the results.


## Importing pandas

I use pandas because it allows me to load the Excel file and work with it as a table.

I also install `openpyxl` because pandas uses it to read and write Excel files.


In [ ]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd  # use pandas to work with tables of data

## Loading the Expert Reviews data

I load the original Expert Reviews Excel file into a pandas DataFrame called `expert`.

At this point I have not cleaned the data yet. I first want to see what the original dataset looks like.


In [ ]:
expert = pd.read_excel("ExpertReviewsClean43LIWC.xlsx")

In [ ]:
# Display the first few rows so I can see what the data looks like
print(expert.head())

                                        url  idvscore            reviewer  \
0  https://www.metacritic.com/movie/bronson     100.0    "Andrew O'Hehir"   
1  https://www.metacritic.com/movie/bronson      90.0        'A.O. Scott'   
2  https://www.metacritic.com/movie/bronson      90.0                None   
3  https://www.metacritic.com/movie/bronson      83.0       'Noel Murray'   
4  https://www.metacritic.com/movie/bronson      80.0   'Joshua Rothkopf'   

   dateP                                                Rev  WC  Analytic  \
0   None   'Bronson owes a little or a lot to Kubrick s ...  25     73.88   
1   None   'Bronson invites you to admire its protagonis...  30     13.07   
2   None   'Whether it s Peterson/Bronson s more theatri...  40     72.69   
3   None   'There are two Bronsons on display here: the ...  39     65.46   
4   None   'Refn has somehow found his way to an authent...  24     88.46   

   Clout  Authentic   Tone  ...  Comma  Colon  SemiC  QMark  Exclam   Dash

In [ ]:
# Check how many rows and columns are in the dataset
expert.shape

(238973, 98)

In [ ]:
# Display all column names in the original dataset
expert.columns

Index(['url', 'idvscore', 'reviewer', 'dateP', 'Rev', 'WC', 'Analytic',
       'Clout', 'Authentic', 'Tone', 'WPS', 'Sixltr', 'Dic', 'function',
       'pronoun', 'ppron', 'i', 'we', 'you', 'shehe', 'they', 'ipron',
       'article', 'prep', 'auxverb', 'adverb', 'conj', 'negate', 'verb', 'adj',
       'compare', 'interrog', 'number', 'quant', 'affect', 'posemo', 'negemo',
       'anx', 'anger', 'sad', 'social', 'family', 'friend', 'female', 'male',
       'cogproc', 'insight', 'cause', 'discrep', 'tentat', 'certain', 'differ',
       'percept', 'see', 'hear', 'feel', 'bio', 'body', 'health', 'sexual',
       'ingest', 'drives', 'affiliation', 'achieve', 'power', 'reward', 'risk',
       'focuspast', 'focuspresent', 'focusfuture', 'relativ', 'motion',
       'space', 'time', 'work', 'leisure', 'home', 'money', 'relig', 'death',
       'informal', 'swear', 'netspeak', 'assent', 'nonflu', 'filler',
       'AllPunc', 'Period', 'Comma', 'Colon', 'SemiC', 'QMark', 'Exclam',
       'Dash', 'Q

## Checking the structure of the dataset

Before changing anything, I check the data types and how many values are present in each column.

This helps me understand the dataset before I start removing or changing anything.


In [ ]:
# Check the structure, non-missing values and data types
expert.info()

<class 'pandas.DataFrame'>
RangeIndex: 238973 entries, 0 to 238972
Data columns (total 98 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   url           238973 non-null  str    
 1   idvscore      238971 non-null  float64
 2   reviewer      238971 non-null  str    
 3   dateP         238971 non-null  str    
 4   Rev           238971 non-null  str    
 5   WC            238973 non-null  int64  
 6   Analytic      238973 non-null  float64
 7   Clout         238973 non-null  float64
 8   Authentic     238973 non-null  float64
 9   Tone          238973 non-null  float64
 10  WPS           238973 non-null  float64
 11  Sixltr        238973 non-null  float64
 12  Dic           238973 non-null  float64
 13  function      238973 non-null  float64
 14  pronoun       238973 non-null  float64
 15  ppron         238973 non-null  float64
 16  i             238973 non-null  float64
 17  we            238973 non-null  float64
 18  you           2

In [ ]:
# Count missing values in each column
expert.isna().sum()

url         0
idvscore    2
reviewer    2
dateP       2
Rev         2
           ..
Dash        0
Quote       0
Apostro     0
Parenth     0
OtherP      0
Length: 98, dtype: int64

## Checking for exact duplicate rows

I check for exact duplicate rows because the same review should not appear twice in the dataset.

First I count the duplicates. Then I look at them so I can see what is being repeated before removing them.


In [ ]:
# Count exact duplicate rows
expert.duplicated().sum()

np.int64(27)

In [ ]:
# Look at the duplicate review records
expert[expert.duplicated(keep=False)][
    ["url", "idvscore", "reviewer", "dateP", "Rev"]
].head(10)

,url,idvscore,reviewer,dateP,Rev
9416,https://www.metacritic.com/movie/bad-samaritan,20.0,'Peter Bradshaw',"'Aug 24, 2018'",'Given the nasty taste in the mouth that the ...
9417,https://www.metacritic.com/movie/bad-samaritan,20.0,'Peter Bradshaw',"'Aug 24, 2018'",'Given the nasty taste in the mouth that the ...
26212,https://www.metacritic.com/movie/kusama---infi...,85.0,'Lisa Jo Sagolla',"'Sep 6, 2018'","'Briskly paced, the film makes for a visually..."
26213,https://www.metacritic.com/movie/kusama---infi...,85.0,'Lisa Jo Sagolla',"'Sep 6, 2018'","'Briskly paced, the film makes for a visually..."
48958,https://www.metacritic.com/movie/spiral-from-t...,46.0,'Jacob Oller',"'May 12, 2021'",'Spiral might have rhetorical wrinkles that s...
48959,https://www.metacritic.com/movie/spiral-from-t...,46.0,'Jacob Oller',"'May 12, 2021'",'Spiral might have rhetorical wrinkles that s...
49979,https://www.metacritic.com/movie/voyagers,50.0,'David Fear',"'Apr 8, 2021'",'This is a pulpy B movie that is dying to be ...
49980,https://www.metacritic.com/movie/voyagers,50.0,'David Fear',"'Apr 8, 2021'",'This is a pulpy B movie that is dying to be ...
52470,https://www.metacritic.com/movie/last-letter-f...,50.0,'Roger Moore',"'Jul 26, 2021'",'The leads make this tolerable.'
52471,https://www.metacritic.com/movie/last-letter-f...,50.0,'Roger Moore',"'Jul 26, 2021'",'The leads make this tolerable.'


## Removing exact duplicates

I remove the exact duplicate rows with `drop_duplicates()`.

I save the cleaned version as `expert_clean` so that I still have the original `expert` DataFrame available if I need to compare it later.


In [ ]:
# Remove exact duplicate rows; the first copy is kept
expert_clean = expert.drop_duplicates()

In [ ]:
# Check the new number of rows and columns
expert_clean.shape

(238946, 98)

In [ ]:
# Check that no exact duplicate rows remain
expert_clean.duplicated().sum()

np.int64(0)

## Checking the rows with missing values

I now look at the rows that contain missing values.

I am especially interested in whether a row is missing the actual review text. A review without review text cannot be used for the language analysis.


In [ ]:
# Display the rows containing missing values
expert_clean[expert_clean.isna().any(axis=1)][
    ["url", "idvscore", "reviewer", "dateP", "Rev", "WC"]
]

,url,idvscore,reviewer,dateP,Rev,WC
41664,https://www.metacritic.com/movie/whiteout,NaN,NaN,NaN,NaN,0
157759,https://www.metacritic.com/movie/micmac---au-f...,NaN,NaN,NaN,NaN,0


## Removing rows without review text

The rows above do not contain actual review text.

I remove rows where `Rev` is missing. I do this specifically because the review text is important for the LIWC variables and for the research question.


In [ ]:
# Remove rows where the actual review text is missing
expert_clean = expert_clean.dropna(subset=["Rev"])

In [ ]:
# Check that review text is no longer missing
expert_clean["Rev"].isna().sum()

np.int64(0)

In [ ]:
# Check the size after removing the rows without review text
expert_clean.shape

(238944, 98)

## Checking the critic scores

The `idvscore` column contains the score associated with each critic review.

I check its distribution and then specifically investigate reviews with a score of 0 before deciding whether they should be removed.


In [ ]:
# Summarise the critic review scores
expert_clean["idvscore"].describe()

count    238944.000000
mean         62.718444
std          20.752745
min           0.000000
25%          50.000000
50%          63.000000
75%          78.000000
max         100.000000
Name: idvscore, dtype: float64

In [ ]:
# Count reviews with an idvscore of 0
(expert_clean["idvscore"] == 0).sum()

np.int64(1678)

In [ ]:
# Inspect some of the reviews with a score of 0
expert_clean[expert_clean["idvscore"] == 0][
    ["url", "idvscore", "reviewer", "dateP", "Rev", "WC"]
].head(20)

,url,idvscore,reviewer,dateP,Rev,WC
472,https://www.metacritic.com/movie/basic,0.0,'Robert Wilonsky',None,'Basic really brings to mind a Travolta film ...,55
494,https://www.metacritic.com/movie/the-real-cancun,0.0,'Marjorie Baumgarten',None,'Shoddy craftsmanship and uninteresting subje...,43
495,https://www.metacritic.com/movie/the-real-cancun,0.0,'Scott Tobias',None,"'Save for the diminished allure of drunk, nak...",18
496,https://www.metacritic.com/movie/the-real-cancun,0.0,'Mark Caro',None,'This movie thrusts you so close to these int...,24
520,https://www.metacritic.com/movie/the-order,0.0,'Owen Gleiberman',None,'The picture is so lethargic that I began to ...,18
595,https://www.metacritic.com/movie/darkness-falls,0.0,'Desson Thomson',None,'A truly awful and extremely loud scareflick.',7
624,https://www.metacritic.com/movie/a-guy-thing,0.0,'Joe Morgenstern',None,'None',1
653,https://www.metacritic.com/movie/view-from-the...,0.0,'Kimberley Jones',None,'The damn thing is boring. Dull as dirt. Desp...,45
654,https://www.metacritic.com/movie/view-from-the...,0.0,'Joe Morgenstern',None,'None',1
693,https://www.metacritic.com/movie/9-dead-gay-guys,0.0,'V.A. Musetto',None,'Offensive and unwatchable.',3


In [ ]:
# Check whether these 0-score reviews are missing key review information
expert_clean[expert_clean["idvscore"] == 0][
    ["idvscore", "reviewer", "dateP", "Rev", "WC"]
].isna().sum()

idvscore    0
reviewer    0
dateP       0
Rev         0
WC          0
dtype: int64

In [ ]:
# Check the word-count distribution for the 0-score reviews
expert_clean[expert_clean["idvscore"] == 0]["WC"].describe()

count    1678.000000
mean       24.470799
std        14.774797
min         1.000000
25%        13.000000
50%        23.000000
75%        34.000000
max        88.000000
Name: WC, dtype: float64

The 0-score rows contain actual review text and word counts, so I keep them.

I do not treat 0 as missing just because it is the lowest possible value in this column. The source data contains real review records with that value, and I do not have enough evidence to change their meaning.


## Checking for placeholder values in the review text

Some values that look missing are actually stored as text.

I check the review text for the exact placeholder `'None'`. This is different from a real pandas missing value, so `isna()` would not find it.


In [ ]:
# Show the most common review-text values
expert_clean["Rev"].value_counts().head(10)

Rev
'None'                                  4555
'One of the best films of the year.'       8
'Fascinating.'                             7
'Riveting.'                                7
'Spellbinding.'                            6
'Compelling.'                              6
'Extraordinary.'                           5
'Mesmerizing.'                             5
'Deadly dull.'                             5
'Dreadful.'                                4
Name: count, dtype: int64

In [ ]:
# Count review rows where the text is exactly the placeholder None
expert_clean["Rev"].astype(str).str.strip().eq("'None'").sum()

np.int64(4555)

In [ ]:
# Store the placeholder review rows so I can inspect them
none_reviews = expert_clean[
    expert_clean["Rev"].astype(str).str.strip().eq("'None'")
]

# Check how many there are
none_reviews.shape

(4555, 98)

In [ ]:
# Check the main fields for the placeholder rows
none_reviews[["idvscore", "reviewer", "dateP", "WC", "posemo", "negemo", "Tone"]].describe()

,idvscore,WC,posemo,negemo,Tone
count,4555.000000,4555.0,4555.0,4555.0,4555.00
mean,61.038858,1.0,0.0,0.0,25.77
std,22.754172,0.0,0.0,0.0,0.00
min,0.000000,1.0,0.0,0.0,25.77
25%,50.000000,1.0,0.0,0.0,25.77
50%,63.000000,1.0,0.0,0.0,25.77
75%,80.000000,1.0,0.0,0.0,25.77
max,100.000000,1.0,0.0,0.0,25.77


## Removing review rows where the review text is `'None'`

There are 4,555 rows where the review text is the placeholder `'None'`.

These are not actual reviews. I remove them because keeping them would make the text and LIWC measures represent the placeholder rather than a real critic review.


In [ ]:
# Remove rows where the review text is the placeholder None
expert_reviews_clean = expert_clean[
    expert_clean["Rev"].astype(str).str.strip() != "'None'"
].copy()

In [ ]:
# Check the size after removing placeholder review text
expert_reviews_clean.shape

(234389, 98)

In [ ]:
# Check that no placeholder review texts remain
expert_reviews_clean["Rev"].astype(str).str.strip().eq("'None'").sum()

np.int64(0)

## Checking the emotional variables

The variables I use from LIWC for this project are `posemo`, `negemo` and `Tone`.

I check their summary statistics and then check whether any values fall outside the expected 0 to 100 range.


In [ ]:
# Summarise the three emotional variables
expert_reviews_clean[["posemo", "negemo", "Tone"]].describe()

,posemo,negemo,Tone
count,234389.000000,234389.000000,234389.000000
mean,5.367127,2.776378,55.067234
std,6.121163,4.480204,40.044771
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,25.770000
50%,4.260000,0.000000,63.690000
75%,7.690000,4.350000,99.000000
max,100.000000,100.000000,99.000000


In [ ]:
# Check for positive-emotion values below 0
(expert_reviews_clean["posemo"] < 0).sum()

np.int64(0)

In [ ]:
# Check for positive-emotion values above 100
(expert_reviews_clean["posemo"] > 100).sum()

np.int64(0)

In [ ]:
# Check for negative-emotion values below 0
(expert_reviews_clean["negemo"] < 0).sum()

np.int64(0)

In [ ]:
# Check for negative-emotion values above 100
(expert_reviews_clean["negemo"] > 100).sum()

np.int64(0)

In [ ]:
# Check for Tone values below 0
(expert_reviews_clean["Tone"] < 0).sum()

np.int64(0)

In [ ]:
# Check for Tone values above 100
(expert_reviews_clean["Tone"] > 100).sum()

np.int64(0)

All of these range checks return 0, so I do not need to change the emotional variables.


## Checking the reviewer names

Some reviewer names are stored as the text `None` instead of as a real missing value.

I keep the review records because they still contain review information. However, I do not want `None` to become a real critic in my `CRITICS` table.


In [ ]:
# Check for actual missing reviewer values
expert_reviews_clean["reviewer"].isna().sum()

np.int64(0)

In [ ]:
# Check for reviewer names stored as the text None
expert_reviews_clean["reviewer"].astype(str).str.strip().eq("None").sum()

np.int64(24879)

In [ ]:
# Select reviews where the reviewer name is stored as None
missing_reviewer = expert_reviews_clean[
    expert_reviews_clean["reviewer"].astype(str).str.strip().eq("None")
]

# Check that these rows still contain real review text
missing_reviewer["Rev"].astype(str).str.strip().eq("None").sum()

np.int64(0)

In [ ]:
# Check the word counts of reviews without a known reviewer
missing_reviewer["WC"].describe()

count    24879.000000
mean        27.456128
std         12.917789
min          1.000000
25%         18.000000
50%         26.000000
75%         35.000000
max        108.000000
Name: WC, dtype: float64

The missing reviewer rows still contain review text, so I keep them.

Later, when I create the `CRITICS` table, I will exclude the missing names from that table. The corresponding reviews can remain in `CRITIC_REVIEWS` with a missing `critic_id`.


## Cleaning and converting the review dates

The original date field is `dateP`.

Some values are the text `None`, while recorded dates contain quotation marks around the date. I first inspect the values and then create a new `review_date` column with proper date values.


In [ ]:
# Look at some values in the original date column
expert_reviews_clean["dateP"].head(20)

0      None
1      None
2      None
3      None
4      None
5      None
6      None
7      None
8      None
9      None
10     None
11     None
12     None
13     None
14     None
15     None
16     None
17     None
18     None
19     None
Name: dateP, dtype: str

In [ ]:
# Count actual missing values in dateP
expert_reviews_clean["dateP"].isna().sum()

np.int64(0)

In [ ]:
# Count date values stored as the text None
expert_reviews_clean["dateP"].astype(str).str.strip().eq("None").sum()

np.int64(100195)

In [ ]:
# Inspect one recorded date exactly as it is stored
expert_reviews_clean["dateP"].iloc[40]

" 'Apr 14, 2011'"

In [ ]:
# Show the exact representation, including quotation marks
repr(expert_reviews_clean["dateP"].iloc[40])

'" \'Apr 14, 2011\'"'

The quotation marks are part of the stored text, so I remove them before converting the values to dates.


In [ ]:
# Convert the recorded dates into real datetime values
# Invalid or missing dates are changed to NaT
expert_reviews_clean["review_date"] = pd.to_datetime(
    expert_reviews_clean["dateP"]
    .astype(str)
    .str.strip()
    .str.strip("'")
    .replace("None", pd.NA),
    format="%b %d, %Y",
    errors="coerce"
)

# Check one converted date
expert_reviews_clean["review_date"].iloc[40]

0    NaT
1    NaT
2    NaT
3    NaT
4    NaT
5    NaT
6    NaT
7    NaT
8    NaT
9    NaT
10   NaT
11   NaT
12   NaT
13   NaT
14   NaT
15   NaT
16   NaT
17   NaT
18   NaT
19   NaT
Name: review_date, dtype: datetime64[us]

In [ ]:
# Count reviews without a recorded review date
expert_reviews_clean["review_date"].isna().sum()

np.int64(100195)

In [ ]:
# Find the earliest recorded review date
expert_reviews_clean["review_date"].min()

Timestamp('2010-10-20 00:00:00')

In [ ]:
# Find the latest recorded review date
expert_reviews_clean["review_date"].max()

Timestamp('2022-03-02 00:00:00')

There are many reviews without a recorded date. I keep those reviews because I cannot invent a date that is not present in the source data.

This means that any later analysis that specifically requires review dates will have to account for the missing dates.


## Checking important fields after cleaning

I now check that the fields I will need are present and do not contain unexpected blank values.

I also check the number of unique movie URLs and reviewers. A movie URL can appear many times because a movie can have many critic reviews, so repeated URLs are expected.


In [ ]:
# Check for missing critic scores
expert_reviews_clean["idvscore"].isna().sum()

np.int64(0)

In [ ]:
# Check for missing word counts
expert_reviews_clean["WC"].isna().sum()

np.int64(0)

In [ ]:
# Check for completely blank review text
expert_reviews_clean["Rev"].astype(str).str.strip().eq("").sum()

np.int64(0)

In [ ]:
# Check for missing movie URLs
expert_reviews_clean["url"].isna().sum()

np.int64(0)

In [ ]:
# Check for blank movie URLs
expert_reviews_clean["url"].astype(str).str.strip().eq("").sum()

np.int64(0)

In [ ]:
# Count unique movie URLs
expert_reviews_clean["url"].nunique()

11362

In [ ]:
# Check whether any URLs contain leading or trailing spaces
expert_reviews_clean["url"].astype(str).str.strip().ne(expert_reviews_clean["url"]).sum()

np.int64(0)

In [ ]:
# Count unique reviewer values before replacing None with a missing value
expert_reviews_clean["reviewer"].nunique()

1179

In [ ]:
# Show the reviewers with the most review records
expert_reviews_clean["reviewer"].value_counts().head(10)

reviewer
None                    24879
'Roger Moore'            3386
'Roger Ebert'            3046
'James Berardinelli'     3046
'Peter Travers'          2947
'Mick LaSalle'           2723
'Ty Burr'                2656
'Owen Gleiberman'        2569
'Peter Rainer'           2404
'Michael Phillips'       2228
Name: count, dtype: int64

## Converting missing reviewer names to proper missing values

I now replace the text `None` with `pd.NA`.

This is important because `None` should not become a critic in the normalized `CRITICS` table.


In [ ]:
# Remove extra spaces and convert the text None into a real missing value
expert_reviews_clean["reviewer"] = (
    expert_reviews_clean["reviewer"]
    .astype(str)
    .str.strip()
    .replace("None", pd.NA)
)

# Check how many reviewer names are now missing
expert_reviews_clean["reviewer"].isna().sum()

In [ ]:
# Check the final size of the cleaned review dataset
expert_reviews_clean.shape

(234389, 99)

In [ ]:
# Check that exact duplicate rows have not been introduced
a = expert_reviews_clean.duplicated().sum()
print(a)

np.int64(0)

## Saving the cleaned Expert Reviews data

I save the cleaned data before splitting it into the database tables.

The CSV backup is useful because it keeps the cleaned review data in one place, including LIWC variables that are not needed in the final database tables.


In [ ]:
# Save the cleaned Expert Reviews data as a CSV file
expert_reviews_clean.to_csv(
    "ExpertReviews_Cleaned.csv",
    index=False
)

# Creating the CRITICS and CRITIC_REVIEWS tables

The main cleaning of the Expert Reviews data is finished.

I now split the cleaned data according to my ERD.

The `CRITICS` table stores each critic once. The `CRITIC_REVIEWS` table stores the individual reviews and uses `critic_id` to connect each review to its critic.


In [ ]:
# Make a copy of the cleaned data for the database tables
expert = expert_reviews_clean.copy()

# Check the number of rows and columns
expert.shape

(234389, 99)

## Keeping the columns needed for my ERD

The cleaned Expert Reviews data still contains many LIWC variables that are not part of my database design.

For the critic tables, I keep the movie URL, reviewer, date, score, review text and the three emotion-related LIWC variables used in the ERD.


In [ ]:
# Keep only the columns needed for the critic tables
expert = expert[
    [
        "url",
        "reviewer",
        "review_date",
        "idvscore",
        "Rev",
        "WC",
        "posemo",
        "negemo",
        "Tone"
    ]
].copy()

# Check the columns I kept
expert.columns.tolist()

(234389, 9)

## Cleaning the reviewer names and review text

I remove extra spaces and quotation marks from the reviewer names and review text.

This makes the values more consistent before I use the reviewer name to create the `CRITICS` table.


In [ ]:
# Remove extra spaces and quotation marks from reviewer names
expert["reviewer"] = (
    expert["reviewer"]
    .astype("string")
    .str.strip()
    .str.strip("'"")
    .str.strip()
)

# Remove extra spaces and quotation marks from review text
expert["Rev"] = (
    expert["Rev"]
    .astype("string")
    .str.strip()
    .str.strip("'"")
    .str.strip()
)

# Check some reviewer names after cleaning
expert["reviewer"].head(10)

In [ ]:
# Check some review texts after cleaning
expert["Rev"].head(5)

0    Bronson owes a little or a lot to Kubrick s Cl...
1    Bronson invites you to admire its protagonist ...
2    Whether it s Peterson/Bronson s more theatrica...
3    There are two Bronsons on display here: the im...
4    Refn has somehow found his way to an authentic...
Name: Rev, dtype: string

## Creating the CRITICS table

A critic can write many reviews, but the critic's name is a property of the critic, not of each individual review.

I therefore create a separate `CRITICS` table containing each known critic only once and give each critic a unique `critic_id`.


In [ ]:
# Keep the reviewer column and remove missing names
critics = expert[["reviewer"]].dropna()

# Remove duplicate reviewer names
critics = critics.drop_duplicates()

# Sort the names so the IDs are assigned consistently
critics = critics.sort_values("reviewer").reset_index(drop=True)

# Create a unique ID for each critic
critics["critic_id"] = range(1, len(critics) + 1)

# Put critic_id first
critics = critics[["critic_id", "reviewer"]]

# Check the first few rows
critics.head()

,critic_id,reviewer
0,1,A.A. Dowd
1,2,A.O. Scott
2,3,Aaron Cutler
3,4,Aaron Hillis
4,5,Aaron Pruner


In [ ]:
# Check the number of critics and columns
critics.shape

(1179, 2)

In [ ]:
# Check that every critic has a unique ID
critics["critic_id"].is_unique

True

In [ ]:
# Check that each critic name appears only once
critics["reviewer"].duplicated().sum()

np.int64(0)

## Adding critic_id to the reviews

The review data still contains the critic's name, while the `CRITICS` table contains the critic's name and its new `critic_id`.

I merge the two tables using `reviewer`. A left join keeps all reviews, including the reviews where the critic name is missing.


In [ ]:
# Add the critic_id to each review using the reviewer name
expert = expert.merge(
    critics,
    on="reviewer",
    how="left"
)

# Check the first few rows after the merge
expert.head()

,url,reviewer,review_date,idvscore,Rev,WC,posemo,negemo,Tone,critic_id
0,https://www.metacritic.com/movie/bronson,Andrew O'Hehir,NaT,100.0,Bronson owes a little or a lot to Kubrick s Cl...,25,0.00,0.00,25.77,70
1,https://www.metacritic.com/movie/bronson,A.O. Scott,NaT,90.0,Bronson invites you to admire its protagonist ...,30,6.67,0.00,99.00,2
2,https://www.metacritic.com/movie/bronson,None,NaT,90.0,Whether it s Peterson/Bronson s more theatrica...,40,0.00,2.50,2.89,886
3,https://www.metacritic.com/movie/bronson,Noel Murray,NaT,83.0,There are two Bronsons on display here: the im...,39,5.13,5.13,25.77,885
4,https://www.metacritic.com/movie/bronson,Joshua Rothkopf,NaT,80.0,Refn has somehow found his way to an authentic...,24,0.00,0.00,25.77,605


In [ ]:
# Check the columns after adding critic_id
expert.columns.tolist()

['url',
 'reviewer',
 'review_date',
 'idvscore',
 'Rev',
 'WC',
 'posemo',
 'negemo',
 'Tone',
 'critic_id']

In [ ]:
# Count reviews that still do not have a critic_id
expert["critic_id"].isna().sum()

np.int64(0)

The reviews without a `critic_id` are the reviews where the source data does not provide a critic name.

I keep those review records because they still contain review text and the variables needed for the analysis. I do not create a fake critic for them.


## Loading the existing MOVIES table

The `CRITIC_REVIEWS` table in my ERD needs a `movie_id`.

The review data contains the movie URL, while the existing `MOVIES` table contains both `movie_id` and `url`. I use the URL to connect each critic review to the correct movie.

I use the existing `movies.csv` created for the project rather than creating a new movie ID system.


In [ ]:
# Load the existing MOVIES table
movies = pd.read_csv("movies.csv")

# Check the first few rows
movies.head()

,movie_id,url,title,metascore,userscore
0,1,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,70,NaN
1,2,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,76,7.7
2,3,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,54,5.8
3,4,https://www.metacritic.com/movie/10-years,10 Years,61,6.9
4,5,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,63,7.5


## Adding movie_id to the critic reviews

I match the critic reviews to the MOVIES table using the movie URL.

I use a left merge so that every critic review remains in the dataset. I then check whether any review failed to receive a `movie_id`.


In [ ]:
# Add movie_id to each review using the movie URL
expert = expert.merge(
    movies[["movie_id", "url"]],
    on="url",
    how="left"
)

# Check the first few rows
expert.head()

,url,reviewer,review_date,idvscore,Rev,WC,posemo,negemo,Tone,critic_id,movie_id
0,https://www.metacritic.com/movie/bronson,Andrew O'Hehir,NaT,100.0,Bronson owes a little or a lot to Kubrick s Cl...,25,0.00,0.00,25.77,70.0,1565
1,https://www.metacritic.com/movie/bronson,A.O. Scott,NaT,90.0,Bronson invites you to admire its protagonist ...,30,6.67,0.00,99.00,2.0,1565
2,https://www.metacritic.com/movie/bronson,<NA>,NaT,90.0,Whether it s Peterson/Bronson s more theatrica...,40,0.00,2.50,2.89,NaN,1565
3,https://www.metacritic.com/movie/bronson,Noel Murray,NaT,83.0,There are two Bronsons on display here: the im...,39,5.13,5.13,25.77,885.0,1565
4,https://www.metacritic.com/movie/bronson,Joshua Rothkopf,NaT,80.0,Refn has somehow found his way to an authentic...,24,0.00,0.00,25.77,605.0,1565


In [ ]:
# Count reviews that did not receive a movie_id
expert["movie_id"].isna().sum()

np.int64(0)

The result is 0, so every critic review was successfully matched to an existing movie.


## Creating the CRITIC_REVIEWS table

I now select only the columns required by the `CRITIC_REVIEWS` table in my ERD.

I also rename the columns `Rev`, `WC` and `Tone` to the lowercase names used in the database.


In [ ]:
# Keep only the columns needed for CRITIC_REVIEWS
critic_reviews = expert[
    [
        "movie_id",
        "critic_id",
        "review_date",
        "idvscore",
        "Rev",
        "WC",
        "posemo",
        "negemo",
        "Tone"
    ]
].copy()

# Rename columns so they match the ERD
critic_reviews = critic_reviews.rename(
    columns={
        "Rev": "rev",
        "WC": "wc",
        "Tone": "tone"
    }
)

# Check the final columns before adding the review ID
critic_reviews.columns.tolist()

['movie_id',
 'critic_id',
 'review_date',
 'idvscore',
 'Rev',
 'WC',
 'posemo',
 'negemo',
 'Tone']

## Creating critic_review_id

Each review needs its own unique ID in the database.

I number the reviews from 1 to the end of the table and put `critic_review_id` at the beginning.


In [ ]:
# Create a unique ID for every critic review
critic_reviews.insert(
    0,
    "critic_review_id",
    range(1, len(critic_reviews) + 1)
)

# Check the first few rows
critic_reviews.head()

,critic_review_id,movie_id,critic_id,review_date,idvscore,rev,wc,posemo,negemo,tone
0,1,1565,70.0,NaT,100.0,Bronson owes a little or a lot to Kubrick s Cl...,25,0.00,0.00,25.77
1,2,1565,2.0,NaT,90.0,Bronson invites you to admire its protagonist ...,30,6.67,0.00,99.00
2,3,1565,NaN,NaT,90.0,Whether it s Peterson/Bronson s more theatrica...,40,0.00,2.50,2.89
3,4,1565,885.0,NaT,83.0,There are two Bronsons on display here: the im...,39,5.13,5.13,25.77
4,5,1565,605.0,NaT,80.0,Refn has somehow found his way to an authentic...,24,0.00,0.00,25.77


In [ ]:
# Check the final column order
critic_reviews.columns.tolist()

['critic_review_id',
 'movie_id',
 'critic_id',
 'review_date',
 'idvscore',
 'rev',
 'wc',
 'posemo',
 'negemo',
 'tone']

## Saving the two CSV files

The two tables are now ready to share with the group.

`critics.csv` contains one row per known critic.

`critic_reviews.csv` contains the individual critic reviews and connects each review to a movie through `movie_id` and to a critic through `critic_id`.


In [ ]:
# Save the CRITICS table as a CSV file
critics.to_csv("critics.csv", index=False)

In [ ]:
# Save the CRITIC_REVIEWS table as a CSV file
critic_reviews.to_csv("critic_reviews.csv", index=False)

In [ ]:
# Check that both CSV files were created
print("critics.csv created:", os.path.exists("critics.csv"))
print("critic_reviews.csv created:", os.path.exists("critic_reviews.csv"))

True

## Final checks

I do a final check of the two tables before sharing them.

I want to make sure the primary IDs are unique and that every review has a valid movie connection. Reviews without a known critic are expected because the source contains missing critic names.


In [ ]:
# Check the final CRITICS table
print("CRITICS shape:", critics.shape)
print(critics.head())

# Check the final CRITIC_REVIEWS table
print("CRITIC_REVIEWS shape:", critic_reviews.shape)
print(critic_reviews.head())

# Check that the primary keys are unique
print("Unique critic IDs:", critics["critic_id"].is_unique)
print("Unique critic review IDs:", critic_reviews["critic_review_id"].is_unique)

# Check the foreign keys
print("Reviews without movie_id:", critic_reviews["movie_id"].isna().sum())
print("Reviews without critic_id:", critic_reviews["critic_id"].isna().sum())
